In [ ]:
!pip install deepirtools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 33.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-man

In [ ]:
import deepirtools
from deepirtools import IWAVE
import torch
import pandas as pd
deepirtools.manual_seed(123)
df = pd.read_csv('/content/ACTG175_data.csv')
df.head()

,pidnum,age,wtkg,hemo,homo,drugs,karnof,oprior,z30,zprior,...,offtrt,cd40,cd420,cd496,r,cd80,cd820,cens,days,arms
0,10056,48,89.8128,0,0,0,100,0,0,1,...,0,422,477,660.0,1,566,324,0,948,2
1,10059,61,49.4424,0,0,0,90,0,1,1,...,0,162,218,NaN,0,392,564,1,1002,3
2,10089,45,88.4520,0,1,1,90,0,1,1,...,1,326,274,122.0,1,2063,1893,0,961,3
3,10093,47,85.2768,0,1,0,100,0,1,1,...,0,287,394,NaN,0,1590,966,0,1166,3
4,10124,43,66.6792,0,1,0,100,0,1,1,...,0,504,353,660.0,1,870,782,0,1090,0


In [ ]:
#Listwise Deletion
df = df.dropna()
# Get the dimensions of the DataFrame
dimensions = df.shape
print(dimensions)

(1342, 27)


In [ ]:
data = df[['cd420', 'cd496', 'cd820']]
data.head()

,cd420,cd496,cd820
0,477,660.0,324
2,274,122.0,1893
4,353,660.0,782
5,339,264.0,1060
6,225,106.0,699


In [ ]:
data.mean()

cd420    381.330104
cd496    328.570790
cd820    939.334575
dtype: float64

In [ ]:
####Standardlize the values of those three variables, it has a mean of zero and a standard deviation of one.
from sklearn.preprocessing import StandardScaler
import pandas as pd
scaler = StandardScaler()

# Fit the scaler to the data and transform it
scaled_data = scaler.fit_transform(data)

# Convert the scaled data back to a DataFrame
data = pd.DataFrame(scaled_data, columns=data.columns)

In [ ]:
data.head()

,cd420,cd496,cd820
0,0.658477,1.898317,-1.356752
1,-0.738732,-1.183169,2.102738
2,-0.194991,1.898317,-0.346907
3,-0.291350,-0.369840,0.266055
4,-1.075990,-1.274812,-0.529914


In [ ]:
numpy_array = data.to_numpy()
print("\nNumPy Array:")
print(numpy_array)


NumPy Array:
[[ 0.65847734  1.89831711 -1.35675185]
 [-0.7387323  -1.18316929  2.1027379 ]
 [-0.19499062  1.89831711 -0.34690717]
 ...
 [ 1.78725754  0.94752391  1.24503139]
 [-0.11927975  1.13080934  0.2241622 ]
 [-1.46142694 -1.72157026  1.98146834]]


In [ ]:
data = torch.tensor(numpy_array, dtype=torch.float32)
print("\nPyTorch Tensor:")
print(data)


PyTorch Tensor:
tensor([[ 0.6585,  1.8983, -1.3568],
        [-0.7387, -1.1832,  2.1027],
        [-0.1950,  1.8983, -0.3469],
        ...,
        [ 1.7873,  0.9475,  1.2450],
        [-0.1193,  1.1308,  0.2242],
        [-1.4614, -1.7216,  1.9815]])


In [ ]:
#Binary matrix indicating measurement structure.

#A matrix of n_items * latent_size. Elements are zero if the corresponding loading is set to zero and one otherwise:

In [ ]:
model = IWAVE(
    model_type = "normal",
    latent_size = 1,# Number of latent variables is 1
    n_items = 3,
    #n_cats = [None, None, None], #Only needed if some items are categorical.
    Q = torch.block_diag(*[torch.ones([3, 1])] * 1),
    device = 'cuda'
    )


NameError: name 'IWAVE' is not defined

In [ ]:
model.fit(data, iw_samples = 5000) #iw_samples can be changed!


Fitting started
Epoch =      676 Iter. =    28401 Cur. loss =    3.94   Intervals no change =  100
Fitting ended in  143.36  seconds


In [ ]:
model.loadings

tensor([[0.9798],
        [0.6774],
        [0.2196]])

In [ ]:
model.cov ##variance of y.

tensor([[1.]])

In [ ]:
model.intercepts

tensor([ 2.6533e-03, -6.8734e-05, -2.2065e-03])

# Estimating Factor Score:

In [ ]:
fs = model.scores(data) #Approximate expected a posteriori (EAP) factor scores given a data set.

In [ ]:
fs.shape

torch.Size([1342, 1])

In [ ]:
fs

tensor([[ 0.6785],
        [-1.1302],
        [-0.0074],
        ...,
        [ 1.6303],
        [-0.1262],
        [-1.5313]])

In [ ]:
fs.mean() ###the factor score looks at the same level with observed variables' values.

tensor(0.0060)

In [ ]:
model.save_model('model_1','/content')

In [ ]:
model.load_model('model_1','/content') #Need to upload the model path file first each time when restarting notebook in colab, then load the fitted model.

In [ ]:
model.intercepts

tensor([ 2.6533e-03, -6.8734e-05, -2.2065e-03])

# Other models with other parameters

Increasing iw_samples decreases the log-likelihood estimator’s bias.

In [ ]:
model = IWAVE(
    model_type = "normal",
    latent_size = 1,# Number of latent variables is 1
    n_items = 3,
    #n_cats = [None, None, None], #Only needed if some items are categorical.
    Q = torch.block_diag(*[torch.ones([3, 1])] * 1),######???Is it correct?
    device = 'cuda'
    )


Initializing model parameters
Initialization ended in  0.22  seconds


In [ ]:
model.fit(data, iw_samples = 10000)


Fitting started
Epoch =      152 Iter. =    14401 Cur. loss =    3.80   Intervals no change =  100
Fitting ended in  35.31  seconds


In [ ]:
model.loadings

tensor([[0.9593],
        [0.6874],
        [0.2203]])

In [ ]:
model.save_model('model_2','/content')

In [ ]:
model.load_model('model_2','/content')

In [ ]:
fs2=model.scores(data)
fs2.mean()

tensor(0.0032)

In [ ]:
##factor score needs to save seperately since ther are calculated seperately.
# Save the tensor to a file
torch.save(fs2, 'fs_2.pth')

In [ ]:
model = IWAVE(
    model_type = "normal",
    latent_size = 1,# Number of latent variables is 1
    n_items = 3,
    #n_cats = [None, None, None], #Only needed if some items are categorical.
    Q = torch.block_diag(*[torch.ones([3, 1])] * 1),######???Is it correct?
    device = 'cuda'
    )
model.fit(data, iw_samples = 8000)


Initializing model parameters
Initialization ended in  0.01  seconds

Fitting started
Epoch =      750 Iter. =    31501 Cur. loss =    4.27   Intervals no change =   19

In [ ]:
fs3=model.scores(data)
fs3.mean()

tensor(-0.0008)

In [ ]:
fs3

tensor([[ 0.6245],
        [-0.6271],
        [ 0.1601],
        ...,
        [ 1.6772],
        [ 0.0431],
        [-1.5241]])

In [ ]:
model.save_model('model_3','/content')

In [ ]:
data3 = {'x': x, 't': t, 'y': y, 'mu_0': mu_0, 'mu_1':mu_1}
data2['t']=data2['t'].reshape(-1,1) #we're just padding one dimensional vectors with an additional dimension
data2['y']=data2['y'].reshape(-1,1)

#rescaling y between 0 and 1 often makes training of DL regressors easier
data2['y_scaler'] = StandardScaler().fit(data2['y'])
data2['ys'] = data2['y_scaler'].transform(data2['y'])